# Network propagation development

The notebook currently covers how results from a AnnData/MuData object can be added to a cpr_graph to setup network-based inference. The general strategy is to:
1. pull out a pd.DataFrame containing feature-level measures of interest along with feature metadata.
2. These are then mapped on the species ids in an sbml_dfs model by shared on ontology, disambiguated (to handle mapping of multiple features to the same s_id), and s_id-indexed results are embedded in the sbml_dfs as a table in species_data
3. attributes of interrest are then passed from the sbml_dfs model into the graph.

This example uses real MuData results but only a small sbml_dfs object which has uniprot but not ENSG identifiers. This makes things easy to work with but a genome-scale graph will need to be used for a real analysis.

Reflecting on the current functionality,

(1) is not too hard but the interface can probably be cleaned up as we should have a function which applies 1-3 in a single call.
(2) is in pretty good shape following a LOT of new functionality being added to napistu-py for handling many-to-one mappings and wide/nested formats for identifiers.
(3) will need some better functionality since the reaction_attrs syntax is pretty cryptic but the core functionality is all there.

Next, steps will be develop basic PPR functionality.

In [1]:
import os

import mudata as md
import numpy as np
import pandas as pd

from napistu import utils as napistu_utils
from napistu.network import net_create
from napistu.network import net_propagation
from napistu.network import data_handling


from napistu.gcs import downloads
from napistu.matching import mount
from napistu.scverse.loading import prepare_anndata_results_df
from napistu.scverse.loading import prepare_mudata_results_df
from napistu.matching.constants import BIND_DICT_OF_WIDE_RESULTS_STRATEGIES_LIST

# setup logging
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# paths
PROJECT_DIR =  os.path.expanduser("~/Desktop/DATA/Forny2023")
SUPPLEMENTAL_DATA_DIR = os.path.join(PROJECT_DIR, "input")
CACHE_DIR = os.path.join(PROJECT_DIR, "cache")
NAPISTU_DATA_DIR = os.path.expanduser("~/Desktop/DATA/napistu_data")

# Define the path to save hyperparameter scan results
MOFA_PARAM_SCAN_MODELS_PATH = os.path.join(CACHE_DIR, "mofa_param_scan_h5mu")
# Final results 
OPTIMAL_MODEL_H5MU_PATH = os.path.join(CACHE_DIR, "mofa_optimal_model.h5mu")

In [2]:
sbml_dfs_path = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "sbml_dfs"
)

identifiers_path = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "identifiers"
)


In [3]:
# ~2 min load
sbml_dfs = napistu_utils.load_pickle(sbml_dfs_path)
species_identifiers = pd.read_csv(identifiers_path, delimiter = "\t")

In [4]:
#net_utils.validate_assets(
#    sbml_dfs = sbml_dfs,
#    cpr_graph = cpr_graph,
#    # TODO - it should really possible for this to be optional
#    precomputed_distances = None,
#    identifiers = species_identifiers
#    )

In [5]:
# lets load the Forny results so we can try adding a few different types of tables to the sbml_dfs
mdata = md.read_h5mu(OPTIMAL_MODEL_H5MU_PATH)

DEBUG:h5py._conv:Creating converter from 3 to 5


## Adding genome-scale datasets

To use an 'omic dataset in Napistu, we want to:
1. mount the dataset on the pathway `sbml_dfs`. This entails:
    - matching systematic identifiers between the dataset and pathway to connect 'omic features to Napistu `species`.
    - resolve many-to-1 mappings (e.g., where 2+ features match the same species).
    - create a table with unique species ids as the index with variable from the dataset.
    - add this to the `species_data` attriute of the `sbml_dfs`. Multiple tables and/or datasets can be added to `species_data`.
2. pass variables from one or more `species_data` tables to a `napistu_graph`'s vertices with `net_create._add_graph_species_attribute`. Variables can be transformed (e.g., to make them non-negative for personalized pagerank) at this point (or this could be done before step (1)).
3. use these verterx attributes for downstream analysis (e.g., using it in the reset_proportional_to parameters of PPR).

Step (1) needs to be adapted depending on how datasets are organized. The currently, supported inputs are:
- `pd.DataFrame` objects which including 1+ systematic identifiers
- `anndata.AnnData` objects where the `var` table provided identifiers, and feature-level summaries come from either the `var`, `varm` or `X` tables.
- `mudata.MuData` objects containing multiple `AnnData` objects where `var` and `varm` attributes can be defined across multiple datasets.

We'll provide an examples using each of these inputs

### Loading results from a pd.DataFrame

In [6]:
SUPPLEMENTAL_DATA_DIR = os.path.join(PROJECT_DIR, "input")
VZ_LMM_RESULTS = {
    "transcriptomics": "diff_exp_lmm_rnaseq_pathwayact_all_annotout.txt",
    "proteomics": "diff_exp_lmm_prot_pathwayact_all_annotout.txt"
}

sideloaded_data_path = {x : os.path.join(SUPPLEMENTAL_DATA_DIR, y) for x, y in VZ_LMM_RESULTS.items()}

assert all([os.path.isfile(x) for x in sideloaded_data_path.values()])

sideloaded_data = {
    x : pd.read_csv(y, delimiter= "\t") for x, y in sideloaded_data_path.items()
}

In [7]:
min(sideloaded_data["proteomics"]["fdr"])

0.177438594834749

In [8]:
for k in sideloaded_data.keys():
    x = sideloaded_data[k][["ensembl", "chi_sq", "pval", "fdr"]]

    mount.bind_wide_results(
        sbml_dfs,
        x,
        f"{k}_loose_data",
        # map columns to Napistu's controlled vocabulary (constants.ONTOLOGIES)
        ontologies = {"ensembl" : "ensembl_gene"},
        species_identifiers = species_identifiers,
        dogmatic = False,
        verbose = True
    )

DEBUG:napistu.matching.species:Validated ontology columns: {'ensembl_gene'}
INFO:napistu.matching.species:Using columns as results: ['chi_sq', 'pval', 'feature_id', 'fdr']
DEBUG:napistu.matching.species:Final long format shape: (14749, 6)
DEBUG:napistu.matching.species:Matching 14749 features to 42421 species for ontology ensembl_gene
INFO:napistu.matching.species:Found 28526 total matches across 1 ontologies
INFO:napistu.matching.species:-1.4% change in feature_ids (14544 vs 14749)
INFO:napistu.matching.species:2617 s_id(s) map to more than one feature_id.
INFO:napistu.matching.species:Examples of s_id mapping to multiple feature_ids (showing up to 3):
s_id       s_name                        
S00000054  UBE2L3                                     [3752, 9356]
S00000105  Ferritin Complex                           [2387, 7544]
S00000106  Golgi-associated Vesicle Cargo    [12302, 13418, 14573]
Name: feature_id, dtype: object
INFO:napistu.matching.species:4848 feature_id(s) map to more th

## Loading Results from an AnnData object

Since the Forny dataset is a multiomics experiment many of the variablges we are interested in will hold a common interpretation across all modalities. For example, the effect size of a term in a regression holds a common meaning as do the loadings from a multi-omic factor analysis (MOFA) decomposition.

But, many datasets will just be a single modality, and even for multiomic datasets we may be interested in exploring the biology of datamodality-specific attributes. An example in this study is the data-modality specific principal component loadings. Since PCA was performed separately on each data modality the principal components will likely be relatively uncorrelated hence it doesn't make much sense to treat the loadings of PCX to one another across modalities. This is definitely the case for this dataset - PC1 of the proteomics data largely reflects a chromatography-driven technical batch effect which is not seen in the transcriptomics data. To more directly explore this proteomics batch effect we can pull PC1 out of its `AnnData` table.

In [9]:
MUDATA_ONTOLOGIES = {
    "transcriptomics" :
        {"ontologies" : ["ensembl_gene"],
         "index_which_ontology" : "ensembl_gene"},
    "proteomics" :
        {"ontologies" : ["uniprot"],
         "index_which_ontology" : "uniprot"}
}

In [10]:

# TO DO - switch this for PCA loadings. Looks like only the PCs were stored in their AnnDatas

anndata_results_df = prepare_anndata_results_df(
    mdata["proteomics"],
    table_type = "layers",
    index_which_ontology = "uniprot",
    results_attrs = ["MMA001", "MMA004", "MMA005"]
)

mount.bind_wide_results(
    sbml_dfs,
    anndata_results_df,
    "proteomics_var_level_results",
    species_identifiers = species_identifiers,
    ontologies = "uniprot"
)

DEBUG:napistu.scverse.loading:_select_results_attrs called with table_type=layers, results_attrs=['MMA001', 'MMA004', 'MMA005']
INFO:napistu.matching.species:Auto-detected ontology columns: {'uniprot'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot'}
INFO:napistu.matching.species:Using columns as results: ['MMA001', 'MMA004', 'feature_id', 'MMA005']
DEBUG:napistu.matching.species:Final long format shape: (4788, 6)
DEBUG:napistu.matching.species:Matching 4788 features to 124813 species for ontology uniprot
INFO:napistu.matching.species:Found 7062 total matches across 1 ontologies


In [11]:
# we can look at the species data created thus far
for k, v in sbml_dfs.species_data.items():
    print(k)
    display(napistu_utils.style_df(v.head(5)))


transcriptomics_loose_data


,chi_sq,pval,fdr,feature_id
s_id,,,,
S00000000,2.337,0.126,1.000,8562
S00000003,0.000,0.983,1.000,4041
S00000005,0.152,0.697,1.000,2583
S00000010,0.426,0.514,1.000,14504
S00000011,0.426,0.514,1.000,14504


proteomics_loose_data


,chi_sq,pval,fdr,feature_id
s_id,,,,
S00000000,0.117,0.733,1.000,2210
S00000005,0.009,0.923,1.000,2037
S00000010,0.965,0.326,1.000,568
S00000011,0.965,0.326,1.000,568
S00000012,1.060,0.303,1.000,661


proteomics_var_level_results


,MMA001,MMA004,MMA005,feature_id
s_id,,,,
S00000000,-0.390,1.863,-0.521,2484
S00000005,-0.823,0.964,-2.002,2303
S00000010,-0.418,0.983,0.626,730
S00000014,-0.018,0.459,0.252,833
S00000052,1.485,0.952,0.508,202


### Loading Results from a MuData object

MuData is a data structure for organizing multiple AnnData objects which can be maninpulated with AnnData-level operations but the combined dataset also has its own attributes which pertain to all data modalities. Napistu provides convenience functions for pulling multi-omic attributes out a MuData object and these can either be stored as a separate attribute for each modality or as a single summary defined over all modalities. The latter workflow may be helpful when combining modalities with non-overlapping ontologies - for example, proteins and metabolites. While, keeping modalities separate may be preferred if multiple modalities would map to the same nodes. For example, working with transcriptomics and proteomics, like in the Forny study, we may want to separately map want to use separate vertex attributes for each modality. This may not be necessary if we are working in "dogmatic" mode where genes, transcripts, and proteins are generally represented as separate nodes, but in non-dogmatic mode these entries are treated equivalently. But, here the network that we are working with was created in non-dogmatic mode ([link](https://github.com/napistu/napistu/blob/26402a440be9d9cb901c1edf371ef0a7e5475e55/dev/create_human_consensus.qmd#L215)).

In [12]:
split_results_tables = prepare_mudata_results_df(
    mdata,
    mudata_ontologies=MUDATA_ONTOLOGIES,
    table_type="varm",
    table_name="LFs", # this would be autodetected
    results_attrs=["LF1", "LF2", "LF3", "LF4", "LF5"],
    table_colnames=[f"LF{i}" for i in range(1, mdata.varm["LFs"].shape[1] + 1)]
)

for k, v in split_results_tables.items():
    print(k)
    display(napistu_utils.style_df(v.head(5)))


DEBUG:napistu.scverse.loading:_select_results_attrs called with table_type=varm, results_attrs=['LF1', 'LF2', 'LF3', 'LF4', 'LF5']
DEBUG:napistu.matching.species:Validated ontology columns: {'ensembl_gene'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot'}


transcriptomics


,ensembl_gene,LF1,LF2,LF3,LF4,LF5
ensembl_gene,,,,,,
ENSG00000187634,ENSG00000187634,-0.132,-0.066,-0.000,-0.106,-0.006
ENSG00000188976,ENSG00000188976,-0.078,0.007,-0.000,-0.092,-0.048
ENSG00000187608,ENSG00000187608,-0.044,0.042,0.000,-0.129,-0.035
ENSG00000188157,ENSG00000188157,-0.017,-0.008,0.000,0.095,-0.198
ENSG00000078808,ENSG00000078808,0.069,-0.001,-0.000,-0.069,-0.021


proteomics


,uniprot,LF1,LF2,LF3,LF4,LF5
uniprot,,,,,,
A0AVF1,A0AVF1,0.259,-0.072,-0.173,0.006,-0.034
A0AVT1,A0AVT1,-0.020,0.027,0.058,-0.001,-0.018
A0FGR8,A0FGR8,0.065,-0.087,-0.017,-0.017,0.006
A1AG_BOVINAlpha-1-acidglycoproteinOS=BostaurusGN=ORM1PE=2SV=1;CONT_Q3SZR3,A1AG_BOVINAlpha-1-acidglycoproteinOS=BostaurusGN=ORM1PE=2SV=1;CONT_Q3SZR3,-0.107,-0.008,0.072,-0.003,0.061
A1L0T0,A1L0T0,-0.005,-0.010,0.238,0.001,0.023


Now, we can can decide how we want to mount these objects on an `SBML_dfs` object. We could either:
- add each modality's results as a separate key-value pair in the species_data attribute
- add them as the same attribute but change the attribute's names to distinguish modalities
- merge them into a single table using the same attribute name for all modalities. This may result in merging of multiple modalities results if they map to the same species.

To handle these different workflows, we can use the `bind_dict_of_wide_results()` function. To better understand their behavior we'll add the same data table using each strategy:

In [13]:
for strategy in BIND_DICT_OF_WIDE_RESULTS_STRATEGIES_LIST:

    mount.bind_dict_of_wide_results(
        sbml_dfs,
        split_results_tables,
        f"{strategy}_results",
        strategy = strategy,
        species_identifiers = species_identifiers,
        # ontologies were already renamed to the controlled vocabulary in prepare_mudata_results_df()
        ontologies = None,
        # ignored because species_identifiers is provided
        dogmatic = False,
        # for clarity; default is True
        inplace = True,
        verbose = False
    )

INFO:napistu.matching.species:Auto-detected ontology columns: {'uniprot', 'ensembl_gene'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot', 'ensembl_gene'}
INFO:napistu.matching.species:Using columns as results: ['LF3', 'LF4', 'LF2', 'LF1', 'feature_id', 'LF5']
DEBUG:napistu.matching.species:Final long format shape: (13922, 8)
DEBUG:napistu.matching.species:Matching 4788 features to 124813 species for ontology uniprot
DEBUG:napistu.matching.species:Matching 9134 features to 42421 species for ontology ensembl_gene
INFO:napistu.matching.species:Found 25537 total matches across 2 ontologies
INFO:napistu.matching.species:Auto-detected ontology columns: {'ensembl_gene'}
DEBUG:napistu.matching.species:Validated ontology columns: {'ensembl_gene'}
INFO:napistu.matching.species:Using columns as results: ['LF3', 'LF4', 'LF2', 'LF1', 'feature_id', 'LF5']
DEBUG:napistu.matching.species:Final long format shape: (9134, 8)
DEBUG:napistu.matching.species:Matching 9134 features to 

In [14]:
results = sbml_dfs.species_data["concatenate_results"]
print(f"Concatenated results; shape: {results.shape}")
display(napistu_utils.style_df(results.head(5)))

results = sbml_dfs.species_data["stagger_results"]
print(f"Staggered results; shape: {results.shape}")
display(napistu_utils.style_df(results.head(5)))

print("Separated results")
for k in split_results_tables.keys():
    species_data_name = f"multiple_keys_results_{k}"
    results = sbml_dfs.species_data[species_data_name]
    print(f"{species_data_name}; shape: {results.shape}")
    display(napistu_utils.style_df(results.head(5)))


Concatenated results; shape: (17023, 6)


,LF1,LF2,LF3,LF4,LF5,feature_id
s_id,,,,,,
S00000000,-0.010,0.063,-0.031,-0.001,0.023,11618
S00000005,-0.019,-0.015,0.033,-0.036,0.016,"11437,1660"
S00000010,0.016,-0.154,0.003,-0.044,-0.012,"8982,9864"
S00000011,-0.102,-0.165,-0.000,-0.083,0.013,8982
S00000012,-0.158,-0.083,0.000,-0.165,0.010,50


Staggered results; shape: (17023, 11)


,LF1_transcriptomics,LF2_transcriptomics,LF3_transcriptomics,LF4_transcriptomics,LF5_transcriptomics,LF1_proteomics,LF2_proteomics,LF3_proteomics,LF4_proteomics,LF5_proteomics,feature_id
s_id,,,,,,,,,,,
S00000000,0.000,0.000,0.000,0.000,0.000,-0.010,0.063,-0.031,-0.001,0.023,11618
S00000005,-0.044,-0.008,0.000,-0.037,0.017,0.025,-0.007,0.033,0.001,-0.001,"11437,1660"
S00000010,-0.025,-0.041,-0.000,-0.021,0.003,0.042,-0.112,0.003,-0.023,-0.015,"8982,9864"
S00000011,-0.102,-0.165,-0.000,-0.083,0.013,0.000,0.000,0.000,0.000,0.000,8982
S00000012,-0.158,-0.083,0.000,-0.165,0.010,0.000,0.000,0.000,0.000,0.000,50


Separated results
multiple_keys_results_transcriptomics; shape: (16520, 6)


,LF1,LF2,LF3,LF4,LF5,feature_id
s_id,,,,,,
S00000005,-0.089,-0.017,0.000,-0.075,0.034,1660
S00000010,-0.102,-0.165,-0.000,-0.083,0.013,8982
S00000011,-0.102,-0.165,-0.000,-0.083,0.013,8982
S00000012,-0.158,-0.083,0.000,-0.165,0.010,50
S00000014,-0.158,-0.083,0.000,-0.165,0.010,50


multiple_keys_results_proteomics; shape: (6975, 6)


,LF1,LF2,LF3,LF4,LF5,feature_id
s_id,,,,,,
S00000000,-0.010,0.063,-0.031,-0.001,0.023,2484
S00000005,0.051,-0.014,0.066,0.002,-0.002,2303
S00000010,0.056,-0.150,0.003,-0.031,-0.020,730
S00000014,-0.017,-0.072,0.027,-0.016,0.001,833
S00000052,0.125,0.098,-0.006,0.003,-0.076,202


Each of these formats could be useful but for our purposes I like the staggered approach because it separates the same attribute across modalities but maintains consistent naming. This will help later to apply the same analysis (personalized pagerank) to each attribute.

Using the same approach we can pull out other attributes with the `.var` attribute being particularly valuable as it stores all of the variable-level statistical summaries.

In [15]:
# now we can add .var attributes from the mdata

VAR_VARS = ["tstat_MMA_urine", "qval_MMA_urine", "tstat_OHCblPlus", "qval_OHCblPlus", "tstat_responsive_to_acute_treatment", "qval_responsive_to_acute_treatment", "tstat_date_freezing", "qval_date_freezing"]

split_results_tables = prepare_mudata_results_df(
    mdata,
    mudata_ontologies=MUDATA_ONTOLOGIES,
    table_type="var",
    results_attrs=VAR_VARS
)

DEBUG:napistu.scverse.loading:_select_results_attrs called with table_type=var, results_attrs=['tstat_MMA_urine', 'qval_MMA_urine', 'tstat_OHCblPlus', 'qval_OHCblPlus', 'tstat_responsive_to_acute_treatment', 'qval_responsive_to_acute_treatment', 'tstat_date_freezing', 'qval_date_freezing']
DEBUG:napistu.matching.species:Validated ontology columns: {'ensembl_gene'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot'}


In [16]:
mount.bind_dict_of_wide_results(
    sbml_dfs,
    split_results_tables,
    "var_level_results",
    strategy = "stagger",
    species_identifiers = species_identifiers,
    verbose = False
)

INFO:napistu.matching.species:Auto-detected ontology columns: {'uniprot', 'ensembl_gene'}
DEBUG:napistu.matching.species:Validated ontology columns: {'uniprot', 'ensembl_gene'}
INFO:napistu.matching.species:Using columns as results: ['qval_responsive_to_acute_treatment_proteomics', 'tstat_OHCblPlus_transcriptomics', 'qval_date_freezing_transcriptomics', 'qval_OHCblPlus_proteomics', 'qval_date_freezing_proteomics', 'qval_MMA_urine_proteomics', 'tstat_MMA_urine_transcriptomics', 'qval_MMA_urine_transcriptomics', 'tstat_responsive_to_acute_treatment_proteomics', 'tstat_responsive_to_acute_treatment_transcriptomics', 'tstat_date_freezing_transcriptomics', 'tstat_MMA_urine_proteomics', 'tstat_date_freezing_proteomics', 'feature_id', 'qval_OHCblPlus_transcriptomics', 'tstat_OHCblPlus_proteomics', 'qval_responsive_to_acute_treatment_transcriptomics']
DEBUG:napistu.matching.species:Final long format shape: (13922, 19)
DEBUG:napistu.matching.species:Matching 4788 features to 124813 species for 

Here, is the final rundown of species_data tables we've added to the `sbml_dfs`:

In [17]:
for k in sbml_dfs.species_data.keys():
    logger.info(f"{k}: {sbml_dfs.species_data[k].columns.tolist()}")

INFO:__main__:transcriptomics_loose_data: ['chi_sq', 'pval', 'fdr', 'feature_id']
INFO:__main__:proteomics_loose_data: ['chi_sq', 'pval', 'fdr', 'feature_id']
INFO:__main__:proteomics_var_level_results: ['MMA001', 'MMA004', 'MMA005', 'feature_id']
INFO:__main__:concatenate_results: ['LF1', 'LF2', 'LF3', 'LF4', 'LF5', 'feature_id']
INFO:__main__:multiple_keys_results_transcriptomics: ['LF1', 'LF2', 'LF3', 'LF4', 'LF5', 'feature_id']
INFO:__main__:multiple_keys_results_proteomics: ['LF1', 'LF2', 'LF3', 'LF4', 'LF5', 'feature_id']
INFO:__main__:stagger_results: ['LF1_transcriptomics', 'LF2_transcriptomics', 'LF3_transcriptomics', 'LF4_transcriptomics', 'LF5_transcriptomics', 'LF1_proteomics', 'LF2_proteomics', 'LF3_proteomics', 'LF4_proteomics', 'LF5_proteomics', 'feature_id']
INFO:__main__:var_level_results: ['tstat_MMA_urine_transcriptomics', 'qval_MMA_urine_transcriptomics', 'tstat_OHCblPlus_transcriptomics', 'qval_OHCblPlus_transcriptomics', 'tstat_responsive_to_acute_treatment_transc

## Adding Attributes to a Graph

Now, we can pass attributes of interest from species_data tables to the Napistu graph object. We can do this either during a graph's creation or after-the-fact. Here we'll demo three approaches:

- adding data during creation with a graph attributes dictionary.
- adding data after creation with a graph attributes dictionary.
- adding data after creation with `data_handling.add_results_table_to_graph()` a more limited, but (relatively) user-friendly approach. 

### Creating an appropriate graph with data attributes.

For many applications we could use the pre-built Napistu graph which is bundled in the human consensus model:

```python
napistu_graph_path  = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "regulatory_graph"
)
napistu_graph = napistu_utils.load_pickle(napistu_graph_path)
```

But, to apply personalized pagerank on a dircted graph we actually want to flip the edges in the graphs so information flows from targets to regulators. So, we'll create an appropriate graph on-the-fly and we'll also add edge weights to it and pass some attributes from species_data to vertices.

In [18]:
def hard_thresholded_nlog10(x):
    # TODO - look like these weren't stored in scientific notation. To avoid -log10(0) -> Inf
    if x == 0:
        return 5
    elif x < 0.2:
        return -np.log10(x)
    else:
        return 1e-10

CUSTOM_TRANSFORMATIONS = {
    # take the absolute value
    "abs" : lambda x: abs(x),
    # -log10[pvalue]
    "nlog10" : lambda x: -np.log10(x),
    # threshold based on loose FDR threshold and then transform   
    "hard_thresholded_nlog10" : hard_thresholded_nlog10,
    "square" : lambda x: x**2
}

GRAPH_ATTRS = {
    "species": {
        "proteomics_chi_sq": {
            "table": "proteomics_loose_data",
            "variable": "chi_sq",
            "trans": "identity",
        },
        "transcriptomics_chi_sq": {
            "table": "transcriptomics_loose_data",
            "variable": "chi_sq",
            "trans": "identity",
        },
        "proteomics_pvalue" : {
            "table": "proteomics_loose_data",
            "variable": "pval",
            "trans": "nlog10",
        },
        "transcriptomics_pvalue" : {
            "table": "transcriptomics_loose_data",
            "variable": "pval",
            "trans": "nlog10",
        },
    },
    "reactions" : {
        "string_wt" : {
            "table" : "string",
            "variable" : "combined_score",
            "trans" : "string_inv"
        }
    }
}

# note that this could have been combined with the LOOSE_GRAPH_ATTRS but we're keeping them separate for clarity
ADD_GRAPH_ATTRS_SPEC = {
    "species": {
        "proteomics_fdr": {
            "table": "proteomics_loose_data",
            "variable": "fdr",
            "trans": "hard_thresholded_nlog10",
        },
        "transcriptomics_fdr": {
            "table": "transcriptomics_loose_data",
            "variable": "fdr",
            "trans": "hard_thresholded_nlog10",
        }
    }
}


In [19]:
# let's create a new graph so we can invert the edges so information flows from targets to regulators

napistu_graph = net_create.process_cpr_graph(
    sbml_dfs,
    reaction_graph_attrs = GRAPH_ATTRS,
    directed = True,
    edge_reversed = True,
    graph_type = "regulatory",
    weighting_strategy = "mixed",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

INFO:root:Constructing network
INFO:napistu.network.net_create:Dropping 2574 reactions with <= 1 reaction species these underspecified reactions may be due to either unrepresented autoregulation and/or removal of cofactors.
INFO:napistu.network.net_create:Organizing all network nodes (compartmentalized species and reactions)
INFO:napistu.network.net_create:Formatting edges as a regulatory graph
INFO:napistu.network.net_create:Formatting 3506776 reactions species as tiered edges.
INFO:napistu.network.net_create:Adding additional attributes to edges, e.g., # of children and parents.
INFO:napistu.network.net_create:Done preparing regulatory graph
INFO:napistu.network.net_create:Adding reversibility and other meta-data from reactions_data
INFO:napistu.network.net_create:Creating reverse reactions for reversible reactions on a directed graph
INFO:napistu.network.net_create:Formatting cpr_graph output
INFO:root:Adding edge weights with an mixed strategy
INFO:napistu.network.net_create:Creati

### Adding data after creation with a `graph_attr` dict

We can add species_data to a graph using the `graph_attr` specification after a network's creation. This is often preferable because we can create, weight, and pickle a graph upfront and then reuse it across many applications where species_data will differ.

In [20]:
# we can add vertex attributes after a graph's creation using this function
# this is handy when we are pulling attributes from a bunch of tables and need to use different transformations
napistu_graph = data_handling._add_graph_species_attribute(
    napistu_graph,
    sbml_dfs,
    species_graph_attrs = ADD_GRAPH_ATTRS_SPEC,
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

INFO:napistu.network.data_handling:Adding meta-data from species_data
INFO:napistu.network.data_handling:Adding new attribute proteomics_fdr to vertices
INFO:napistu.network.data_handling:Adding new attribute transcriptomics_fdr to vertices


### Adding results with `add_results_table_to_graph`

Adding attributes to graphs with the `graph_spec` dict is powerful and flexible but its not particular user friendly. It may also be a pain when we want to pass many attributes from a species_data table onto a graph. In these cases we can use the `data_handling.add_results_table_to_graph` which selects one or more attributes from a table, by string name, list, dictionary (for renaming), or regular expression and then applies a uniform transformation on them.

In [21]:
# add attributes an AnnData-level table
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "MMA",
    table_name = "proteomics_var_level_results",
    transformation = "square",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

# add results from the MuData-level mvar table
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "LF",
    table_name = "stagger_results",
    transformation = "square",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

# add results from the MuData-level var table - regression results
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "qval",
    table_name = "var_level_results",
    transformation = "hard_thresholded_nlog10",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "tstat",
    table_name = "var_level_results",
    transformation = "abs",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

DEBUG:napistu.network.data_handling:Loading table proteomics_var_level_results from species_data
DEBUG:napistu.network.data_handling:Creating a mapping of attributes to add
INFO:napistu.network.data_handling:Adding meta-data from species_data
INFO:napistu.network.data_handling:Adding new attribute MMA001 to vertices
INFO:napistu.network.data_handling:Adding new attribute MMA004 to vertices
INFO:napistu.network.data_handling:Adding new attribute MMA005 to vertices
DEBUG:napistu.network.data_handling:Loading table stagger_results from species_data
DEBUG:napistu.network.data_handling:Creating a mapping of attributes to add
INFO:napistu.network.data_handling:Adding meta-data from species_data
INFO:napistu.network.data_handling:Adding new attribute LF1_transcriptomics to vertices
INFO:napistu.network.data_handling:Adding new attribute LF2_transcriptomics to vertices
INFO:napistu.network.data_handling:Adding new attribute LF3_transcriptomics to vertices
INFO:napistu.network.data_handling:Add

## Network Propagation

Here we'll implement a workflow for applying network propagation to a cpr_graph's vertex attributes.

In [22]:
annotated_vertices = napistu_graph.get_vertex_dataframe()

# find valid attributes - numeric + 1+ non-zero values
invalid_attributes = [x for x in annotated_vertices.columns if annotated_vertices[x].dtype not in ["float64", "int64"] or annotated_vertices[x].nunique() == 1]
invalid_attributes

['name', 'node_name', 'node_type']

In [23]:
valid_attributes = [x for x in annotated_vertices.columns if x not in invalid_attributes]
for attr in valid_attributes:
    del napistu_graph.vs[attr]

In [24]:
netprop_results = dict()

valid_attributes = [x for x in annotated_vertices.columns if x not in invalid_attributes]
for attribute in valid_attributes:

    try:
        netprop_results[attribute] = net_propagation.personalized_pagerank_by_attribute(
            napistu_graph,
            attribute
        )
    except Exception as e:
        print(f"Error with attribute {attribute}: {e}")
        continue

napistu_utils.style_df(netprop_results[list(netprop_results.keys())[0]].head())

ValueError: Vertex attribute 'proteomics_fdr' is missing for all vertices.

In [55]:
# combine all results into a single pd.DataFrame
reorganized_results = dict()
for k, v in netprop_results.items():
    v["attribute"] = k
    v = v.drop(columns = [k])
    reorganized_results[k] = v

reorganized_results = pd.concat(list(reorganized_results.values()))

### Biological Interpretation

To select for interesting features, we can compare PPR scores to a suitable null distribution. The currently implemented way of doing this to compare:
    - **pagerank_by_attribute** the personalized pagerank scores of vertices when reset probability is proportional to a vertex attribute
    - **pagerank_unifrom** the peronsalized ragerank scores when reset probability is Unfiform over vertices with a non-zero value of the vertex attribute.

This can help to address connectivity biases or if measured vertices show up in a certain region of the network. For example, we may expect metabolites to be nearby other metabolites.

In [ ]:
PPR_SCORE_RATIO = 5

ppr_enrichments = (
    reorganized_results
    .assign(score_ratio=lambda x: x.pagerank_by_attribute/x.pagerank_uniform)
    .sort_values("score_ratio", ascending=False)
).query("score_ratio > @PPR_SCORE_RATIO")

ppr_enrichments.value_counts("attribute")

In [ ]:
stat_results = (
    ppr_enrichments.loc[ppr_enrichments["attribute"].str.match("^tstat")]
    .merge(annotated_vertices[["name", "node_name", "node_type"]], left_on = "name", right_on = "name", how = "left")
)[["attribute","node_name", "node_type", "score_ratio"]]

for k, v in stat_results.groupby("attribute"):
    print(k)
    display(napistu_utils.style_df(v.head(10)))


In [ ]:
stat_results = (
    ppr_enrichments.loc[ppr_enrichments["attribute"].str.match("^qval")]
    .merge(annotated_vertices[["name", "node_name", "node_type"]], left_on = "name", right_on = "name", how = "left")
)[["attribute","node_name", "node_type", "score_ratio"]]

for k, v in stat_results.groupby("attribute"):
    print(k)
    display(napistu_utils.style_df(v.head(10)))
